# NeuralUX — UX Brain Analyzer

Prediz ativação de regiões cerebrais a partir de screenshots de interface usando representações intermediárias de CLIP ViT-L/14.

**Abordagem:** Diferentes camadas de redes neurais profundas (DNNs) correlacionam com diferentes regiões do córtex visual e cognitivo (Yamins et al., 2014; Schrimpf et al., 2018). Extraímos hidden states e attention maps de cada camada do ViT e mapeamos para 8 ROIs cerebrais.

**Antes de rodar:**
1. `Runtime → Change runtime type → GPU (T4 é suficiente)`
2. Configure os tokens abaixo
3. Rode as células em ordem

In [ ]:
# =============================================
# CONFIGURAÇÃO
# =============================================
import os
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "").strip()  # defina no ambiente do Colab
GIST_ID = "397ef638ef931f3ace318e891a741312"  # Gist que o dashboard lê


In [ ]:
%%capture
!pip install -q transformers torch torchvision gradio opencv-python-headless Pillow matplotlib numpy requests

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {vram:.1f} GB")
else:
    print("Nenhuma GPU — rodando em CPU (mais lento)")

print(f"Device: {device}")

In [ ]:
from transformers import CLIPModel, CLIPProcessor

print("Carregando CLIP ViT-L/14...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14", attn_implementation='eager').to(device).eval()
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
print("Modelo pronto!")

In [ ]:
import numpy as np
import cv2
from PIL import Image
import os # Adicionado para o teste automatico

# =============================================
# PROBES SEMANTICOS — embeddings de texto CLIP
# para medir similaridade com conceitos especificos
# =============================================
PROBES = {
    "text":     ["text", "words", "labels", "typography", "reading", "headline"],
    "faces":    ["human face", "portrait", "avatar", "person", "profile photo"],
    "layout":   ["grid layout", "organized structure", "navigation menu", "sidebar"],
    "motion":   ["animation", "movement", "dynamic", "transition", "scrolling"],
    "decision": ["button", "call to action", "form", "checkout", "purchase", "subscribe"],
    "semantic": ["meaningful content", "information", "data visualization", "icons with meaning"],
}

probe_embeddings = {}
for key, texts in PROBES.items():
    inputs = clip_processor(text=texts, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = clip_model.text_model(**inputs)
        emb = clip_model.text_projection(outputs.pooler_output)
        emb = emb / emb.norm(dim=-1, keepdim=True)
        probe_embeddings[key] = emb.mean(dim=0)

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

print("Probes semanticos prontos!")


# =============================================
# FUNCOES AUXILIARES
# =============================================
def layer_magnitude(hidden_states, start, end):
    """Magnitude media de ativacao nas camadas [start:end]"""
    # Certifica que temos hidden_states suficientes
    if not hidden_states or start >= len(hidden_states) or end > len(hidden_states):
        return 0.0 # Retorna 0.0 se nao houver camadas ou indices invalidos
    
    layers_to_stack = [hidden_states[i][:, 1:, :] for i in range(start, end) if hidden_states[i] is not None]
    if not layers_to_stack:
        return 0.0
    layers = torch.stack(layers_to_stack)
    return layers.norm(dim=-1).mean().item()

def get_image_embedding(pil_image):
    """Extrai embedding de imagem normalizado via CLIP"""
    inputs = clip_processor(images=pil_image, return_tensors="pt").to(device)
    outputs = clip_model.vision_model(**inputs)
    emb = clip_model.visual_projection(outputs.pooler_output)
    return emb / emb.norm(dim=-1, keepdim=True)

def clip_similarity(image_features, probe_key):
    """Similaridade cosseno entre imagem e probe de texto"""
    probe = probe_embeddings[probe_key].unsqueeze(0)
    probe = probe / probe.norm(dim=-1, keepdim=True)
    return (image_features @ probe.T).item()


# =============================================
# FUNCAO DE EXTRACAO DE FEATURES ROBUSTA
# =============================================
def extract_clip_features(pil_image):
    inputs = clip_processor(images=pil_image, return_tensors="pt").to(device)
    vision_out = clip_model.vision_model(
        **inputs,
        output_hidden_states=True,
        output_attentions=True,
    )
    
    # Handle cases where hidden_states or attentions might be None themselves
    raw_hidden_states = vision_out.hidden_states if vision_out.hidden_states is not None else ()
    raw_attentions = vision_out.attentions if vision_out.attentions is not None else ()

    hs = [h for h in raw_hidden_states if h is not None]
    attn = [a for a in raw_attentions if a is not None]

    # Embedding de imagem via projection layer
    img_features = clip_model.visual_projection(vision_out.pooler_output)
    img_features = img_features / img_features.norm(dim=-1, keepdim=True)
    return hs, attn, img_features


# =============================================
# SCORING POR REGIAO CEREBRAL
# Cada funcao combina features de DNN + analise
# de imagem para estimar ativacao na ROI.
# =============================================

def score_v1(hidden_states, img_cv, n_layers):
    """V1 — Cortex visual primario: bordas, contraste, frequencia espacial.
    Camadas iniciais do ViT correlacionam com V1 (Schrimpf et al., 2018)."""
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    edge_density = cv2.Canny(gray, 50, 150).mean() / 255.0
    contrast = gray.astype(float).std() / 80.0
    laplacian_var = min(cv2.Laplacian(gray, cv2.CV_64F).var() / 2000.0, 1.0)
    # Usar indices relativos para early layers
    early_norm = min(layer_magnitude(hidden_states, 1, n_layers // 4) / 15.0, 1.0)
    raw = 0.25 * edge_density + 0.25 * contrast + 0.2 * laplacian_var + 0.3 * early_norm
    return float(np.clip(raw * 110, 15, 95))


def score_ffa(hidden_states, img_cv, image_features, n_layers):
    """FFA — Area fusiforme de faces: deteccao facial + features mid-ventral.
    Camadas intermediarias (8-14) correspondem ao stream ventral (Yamins et al., 2014)."""
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(30, 30))
    total_pixels = gray.shape[0] * gray.shape[1]
    face_area = sum(w * h for (_, _, w, h) in faces) / total_pixels if len(faces) > 0 else 0
    face_score = min(len(faces) * 0.2 + face_area * 3, 1.0)
    clip_face = (clip_similarity(image_features, "faces") + 1) / 2
    # Usar indices relativos para mid layers
    mid_norm = min(layer_magnitude(hidden_states, n_layers // 3, n_layers * 2 // 3) / 15.0, 1.0)
    raw = 0.4 * face_score + 0.35 * clip_face + 0.25 * mid_norm
    return float(np.clip(raw * 105, 10, 95))


def score_ppa(hidden_states, img_cv, image_features):
    """PPA — Area parahipocampal: layout e estrutura espacial.
    Detecta linhas horizontais/verticais e regularidade espacial."""
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, 80, minLineLength=50, maxLineGap=10)
    hv_lines = 0
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            angle = abs(np.arctan2(y2 - y1, x2 - x1) * 180 / np.pi)
            if angle < 15 or angle > 165 or 75 < angle < 105:
                hv_lines += 1
    line_score = min(hv_lines / 30.0, 1.0)
    clip_layout = (clip_similarity(image_features, "layout") + 1) / 2
    h, w = gray.shape
    bs = max(h, w) // 8
    if bs > 0:
        blocks = [gray[i:i+bs, j:j+bs].mean() for i in range(0, h - bs, bs) for j in range(0, w - bs, bs)]
        regularity = 1.0 - min(np.std(blocks) / 60.0, 1.0) if blocks else 0.5
    else:
        regularity = 0.5
    raw = 0.35 * line_score + 0.35 * clip_layout + 0.3 * regularity
    return float(np.clip(raw * 110, 15, 95))


def score_v5(hidden_states, img_cv, image_features):
    """V5/MT — Area de movimento: gradientes direcionais e dinamismo implicito.
    Para imagens estaticas, mede elementos que sugerem movimento."""
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY).astype(float)
    sx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=5)
    sy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=5)
    grad_energy = min(np.sqrt(sx**2 + sy**2).mean() / 100.0, 1.0)
    angles = np.arctan2(sy, sx) * 180 / np.pi
    diag = (((angles > 30) & (angles < 60)) | ((angles > 120) & (angles < 150))).mean()
    clip_motion = (clip_similarity(image_features, "motion") + 1) / 2
    hue_std = 0
    if len(img_cv.shape) == 3:
        hue_std = cv2.cvtColor(img_cv, cv2.COLOR_BGR2HSV)[:, :, 0].std() / 90.0
    raw = 0.3 * grad_energy + 0.2 * diag * 3 + 0.3 * clip_motion + 0.2 * min(hue_std, 1.0)
    return float(np.clip(raw * 100, 10, 90))


def score_ips(attentions, img_cv, n_attentions):
    """IPS — Sulco intraparietal: atencao visual e foco.
    Usa attention maps do ViT — baixa entropia = atencao focada = IPS alto."""

    # Fallback se nao houver attentions
    if not attentions or n_attentions == 0:
        return float(np.clip(50.0, 15, 95)) # Pontuacao padrao se nao houver attentions

    # Usar indices relativos para late attentions
    # As ultimas 1/4 partes das camadas de atencao
    start_attn_idx = max(0, n_attentions - n_attentions // 4)
    late_attn = torch.stack([attentions[i] for i in range(start_attn_idx, n_attentions) if attentions[i] is not None])

    if late_attn.nelement() == 0: # Verifica se o tensor esta vazio apos empilhar
        return float(np.clip(50.0, 15, 95))

    avg_attn = late_attn.mean(dim=(0, 2))[:, 0, 1:]
    if avg_attn.nelement() == 0:
        return float(np.clip(50.0, 15, 95))

    probs = avg_attn / avg_attn.sum(dim=-1, keepdim=True)
    entropy = -(probs * torch.log(probs + 1e-10)).sum(dim=-1).mean().item()
    max_entropy = np.log(probs.shape[-1])
    focus = 1.0 - (entropy / max_entropy) if max_entropy > 0 else 0.0 # Evita divisao por zero

    topk_val = probs.topk(k=min(10, probs.shape[-1]), dim=-1).values
    topk = topk_val.sum(dim=-1).mean().item() if topk_val.nelement() > 0 else 0.0

    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    sal = np.abs(gray.astype(float) - gray.mean())
    sal_norm = min(sal.std() / max(sal.mean() + 1e-10, 1.0) / 2.0, 1.0)
    raw = 0.35 * focus * 1.5 + 0.35 * topk + 0.3 * sal_norm
    return float(np.clip(raw * 100, 15, 95))


def score_broca(hidden_states, img_cv, image_features, n_layers):
    """Broca — Area de linguagem: processamento de texto e labels.
    Combina CLIP text-probe com deteccao de regioes textuais (MSER)."""
    clip_text = (clip_similarity(image_features, "text") + 1) / 2
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    mser = cv2.MSER_create()
    regions, _ = mser.detectRegions(gray)
    text_regions = 0
    for region in regions:
        x, y, w, h = cv2.boundingRect(region.reshape(-1, 1, 2))
        aspect = w / max(h, 1)
        area = w * h
        if 0.1 < aspect < 10 and 50 < area < 5000:
            text_regions += 1
    text_density = min(text_regions / 200.0, 1.0)
    # Usar indices relativos para late layers
    late_norm = min(layer_magnitude(hidden_states, n_layers * 2 // 3, n_layers - 1) / 15.0, 1.0)
    raw = 0.4 * clip_text + 0.35 * text_density + 0.25 * late_norm
    return float(np.clip(raw * 110, 10, 95))


def score_pfc(hidden_states, image_features, n_layers):
    """PFC — Cortex pre-frontal: tomada de decisao e memoria de trabalho.
    Diversidade de features nas camadas finais indica carga cognitiva."""
    clip_decision = (clip_similarity(image_features, "decision") + 1) / 2

    if not hidden_states or n_layers == 0:
        return float(np.clip(50.0, 15, 95))

    late_features = hidden_states[n_layers - 1][:, 1:, :] if n_layers > 0 and hidden_states[n_layers - 1] is not None else torch.tensor([])
    diversity = min(late_features.std(dim=1).mean().item() / 5.0, 1.0) if late_features.nelement() > 0 else 0.0

    # Usar indices relativos para as ultimas camadas
    start_cls_idx = max(0, n_layers - n_layers // 4)
    last_layers_cls = [hidden_states[i][:, 0, :] for i in range(start_cls_idx, n_layers) if hidden_states[i] is not None and hidden_states[i].shape[1] > 0]

    complexity = 0.0
    if last_layers_cls:
        stacked_cls = torch.stack(last_layers_cls)
        complexity = min(stacked_cls.std(dim=0).mean().item() / 3.0, 1.0) if stacked_cls.nelement() > 0 else 0.0

    raw = 0.4 * clip_decision + 0.3 * diversity + 0.3 * complexity
    return float(np.clip(raw * 105, 15, 95))


def score_semantic(hidden_states, image_features, n_layers):
    """Semantica — Compreensao de contexto e significado.
    Magnitude e ganho semantico entre camadas iniciais e finais."""
    clip_sem = (clip_similarity(image_features, "semantic") + 1) / 2

    if not hidden_states or n_layers == 0:
        return float(np.clip(50.0, 15, 95))

    final_cls = hidden_states[n_layers - 1][:, 0, :] if n_layers > 0 and hidden_states[n_layers - 1] is not None and hidden_states[n_layers - 1].shape[1] > 0 else torch.tensor([])
    sem_mag = min(final_cls.norm(dim=-1).mean().item() / 25.0, 1.0) if final_cls.nelement() > 0 else 0.0

    # Usar indices relativos para early e late CLS tokens
    early_cls = hidden_states[n_layers // 4][:, 0, :] if n_layers // 4 < n_layers and hidden_states[n_layers // 4] is not None and hidden_states[n_layers // 4].shape[1] > 0 else torch.tensor([])
    late_cls = hidden_states[n_layers - 2][:, 0, :] if n_layers - 2 >= 0 and hidden_states[n_layers - 2] is not None and hidden_states[n_layers - 2].shape[1] > 0 else torch.tensor([])

    gain = 0.0
    if early_cls.nelement() > 0 and late_cls.nelement() > 0:
        gain = min((late_cls - early_cls).norm(dim=-1).mean().item() / 20.0, 1.0)

    raw = 0.35 * clip_sem + 0.35 * sem_mag + 0.3 * gain
    return float(np.clip(raw * 105, 15, 95))


# =============================================
# PREDICAO COMPLETA
# =============================================
@torch.no_grad()
def predict_brain_activation(image_path):
    """Prediz ativacao cerebral a partir de um screenshot de interface."""
    pil_image = Image.open(image_path).convert("RGB")
    img_cv = cv2.imread(image_path)
    if img_cv is None:
        # Fallback para caso cv2.imread falhe (e.g., caminho com caracteres especiais ou nao-ascii)
        img_cv = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)

    hs, attn, img_features = extract_clip_features(pil_image)

    n_layers = len(hs) # Numero real de hidden states apos filtragem
    n_attentions = len(attn) # Numero real de attention maps apos filtragem

    scores = {
        "Visual primário (V1)":       score_v1(hs, img_cv, n_layers),
        "Faces / avatares (FFA)":     score_ffa(hs, img_cv, img_features, n_layers),
        "Layouts / cenas (PPA)":      score_ppa(hs, img_cv, img_features),
        "Movimento / animação (V5)":  score_v5(hs, img_cv, img_features),
        "Atenção visual (IPS)":       score_ips(attn, img_cv, n_attentions),
        "Texto / labels (Broca)":     score_broca(hs, img_cv, img_features, n_layers),
        "Decisão / memória (PFC)":    score_pfc(hs, img_features, n_layers),
        "Semântica / contexto":       score_semantic(hs, img_features, n_layers),
    }
    return scores


print("Pipeline de analise pronto!")

# =============================================
# TESTE AUTOMATICO (NOVA ADICAO)
# =============================================
print("\nExecutando teste automatico com imagem dummy...")
try:
    # Cria uma imagem dummy (e.g., uma imagem preta 224x224)
    dummy_image_path = "/tmp/dummy_image.png"
    Image.new('RGB', (224, 224), color = 'black').save(dummy_image_path)

    test_scores = predict_brain_activation(dummy_image_path)
    print("Teste automatico concluido com sucesso. Scores:")
    for k, v in test_scores.items():
        print(f"  {k}: {v:.2f}")
except Exception as e:
    print(f"Erro durante o teste automatico: {e}")
finally:
    # Limpa a imagem dummy
    if os.path.exists(dummy_image_path):
        os.remove(dummy_image_path)


In [ ]:
import gradio as gr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import tempfile, time, json, requests, os, sys
from pathlib import Path
import numpy as np

# =============================================
# INTEGRACAO NEUROSCORE V2 (repo local)
# =============================================
def _configure_repo_imports():
    candidates = [
        Path.cwd(),
        Path.cwd() / "neuralux-redirect",
        Path("/content/neuralux-redirect"),
    ]
    for base in candidates:
        if (base / "backend" / "neuroscore_v2.py").exists():
            if str(base) not in sys.path:
                sys.path.insert(0, str(base))
            return str(base)
    return None

REPO_BASE = _configure_repo_imports()
HAS_NEUROSCORE_V2 = False
analyze_media = None

try:
    from backend.neuroscore_v2 import analyze_media as _analyze_media
    analyze_media = _analyze_media
    HAS_NEUROSCORE_V2 = True
    print(f"NeuroScore v2 carregado de: {REPO_BASE or Path.cwd()}")
except Exception as e:
    print(f"NeuroScore v2 indisponivel no notebook: {e}")
    print("Fallback para pipeline antigo (predict_brain_activation).")


# =============================================
# GRAFICO
# =============================================
def gerar_grafico(scores):
    nomes = [k.split("(")[0].strip() for k in scores]
    vals = list(scores.values())
    cores = ["#D85A30" if v >= 65 else "#EF9F27" if v >= 40 else "#888780" for v in vals]
    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.barh(nomes, vals, color=cores, height=0.55)
    ax.set_xlim(0, 100)
    ax.set_xlabel("Nivel de ativacao (0-100)")
    ax.set_title("Atividade cerebral por regiao", fontweight="bold", fontsize=12)
    ax.axvline(65, color="#D85A30", linestyle="--", alpha=0.35)
    ax.axvline(40, color="#EF9F27", linestyle="--", alpha=0.35)
    for bar, v in zip(bars, vals):
        ax.text(v + 1, bar.get_y() + bar.get_height() / 2, f"{v:.0f}%", va="center", fontsize=8)
    p1 = mpatches.Patch(color="#D85A30", label="Alta (>=65)")
    p2 = mpatches.Patch(color="#EF9F27", label="Media (40-64)")
    p3 = mpatches.Patch(color="#888780", label="Baixa (<40)")
    ax.legend(handles=[p1, p2, p3], fontsize=8, loc="lower right")
    plt.tight_layout()
    tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    plt.savefig(tmp.name, dpi=140, bbox_inches="tight")
    plt.close()
    return tmp.name


# =============================================
# RELATORIO
# =============================================
def gerar_relatorio(scores, tipo_entrada="imagem estatica"):
    scores_txt = "\n".join([f"- {r}: {s:.1f}/100" for r, s in scores.items()])
    prompt = f"""Voce e um especialista senior em neurociencia cognitiva aplicada a UX design.

O modelo analisou uma {tipo_entrada} de interface e previu as seguintes ativacoes cerebrais (0-100):

{scores_txt}

CONTEXTO:
- Scores >= 65 = ativacao ALTA
- Scores 40-64 = ativacao MEDIA
- Scores < 40 = ativacao BAIXA
- Para imagem estatica, V5 reflete elementos que sugerem dinamismo, nao movimento real.

Gere um relatorio em portugues com:

## DIAGNOSTICO GERAL
3-4 frases sobre o padrao de ativacao e o estado cognitivo predominante.

## PONTOS FORTES
3 pontos fortes (regiao, score, significado neurologico, por que e positivo para UX).

## PONTOS DE ATENCAO
3 pontos de atencao (regiao, score, o que indica, risco para experiencia).

## RECOMENDACOES DE UX
4 recomendacoes concretas (titulo, justificativa neurologica, como implementar).

## SCORE GERAL
Nota de 0 a 10 com justificativa.

Seja direto e tecnico mas acessivel."""

    try:
        api_key = os.environ.get("ANTHROPIC_API_KEY", "").strip()
        if not api_key:
            raise RuntimeError("ANTHROPIC_API_KEY nao definido no ambiente.")

        payload = {
            "model": "claude-3-5-sonnet-20241022",
            "max_tokens": 900,
            "temperature": 0.3,
            "messages": [{"role": "user", "content": prompt}],
        }
        headers = {
            "x-api-key": api_key,
            "anthropic-version": "2023-06-01",
            "content-type": "application/json",
        }
        resp = requests.post(
            "https://api.anthropic.com/v1/messages",
            headers=headers,
            json=payload,
            timeout=45,
        )
        resp.raise_for_status()
        data = resp.json()
        partes = [p.get("text", "") for p in data.get("content", []) if p.get("type") == "text"]
        texto = "\n".join([p for p in partes if p]).strip()
        if not texto:
            raise RuntimeError("Resposta vazia do Claude.")
        return texto
    except Exception as e:
        # Fallback: relatorio baseado em regras
        linhas = ["## DIAGNOSTICO GERAL\n"]
        vals = list(scores.values())
        media = np.mean(vals) if vals else 0
        linhas.append(f"A interface apresenta ativacao media de {media:.0f}/100 nas regioes analisadas.\n")
        top = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        if len(top) >= 2:
            linhas.append(f"Maior ativacao em **{top[0][0]}** ({top[0][1]:.0f}) e **{top[1][0]}** ({top[1][1]:.0f}).")
            linhas.append(f"Menor ativacao em **{top[-1][0]}** ({top[-1][1]:.0f}).\n")
        linhas.append("## SCORES POR REGIAO\n")
        for roi, val in scores.items():
            nivel = "ALTA" if val >= 65 else "MEDIA" if val >= 40 else "BAIXA"
            linhas.append(f"- **{roi}**: {val:.0f}/100 [{nivel}]")
        linhas.append(f"\n*Erro ao gerar analise com IA: {e}*")
        return "\n".join(linhas)


# =============================================
# EXTRAIR PATH DO ARQUIVO GRADIO
# =============================================
def extrair_path(arquivo):
    if arquivo is None:
        return None
    if isinstance(arquivo, str):
        return arquivo
    if isinstance(arquivo, dict):
        return arquivo.get("path") or arquivo.get("name")
    if hasattr(arquivo, "name"):
        return arquivo.name
    return str(arquivo)


def _is_video(path):
    ext = Path(path).suffix.lower()
    return ext in {".mp4", ".mov", ".mkv", ".webm", ".avi", ".m4v"}


def _to_builtin(obj):
    if isinstance(obj, dict):
        return {str(k): _to_builtin(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_to_builtin(v) for v in obj]
    if isinstance(obj, tuple):
        return [_to_builtin(v) for v in obj]
    if isinstance(obj, np.generic):
        return obj.item()
    return obj


def _run_analysis(path):
    tipo = "video" if _is_video(path) else "imagem estatica"

    if HAS_NEUROSCORE_V2 and analyze_media is not None:
        t0 = time.time()
        result = analyze_media(path, transcript=None)
        if not isinstance(result, dict):
            result = {}
        elapsed = time.time() - t0
        result["elapsed"] = float(result.get("elapsed", elapsed))
        scores = result.get("scores", {})
        if not isinstance(scores, dict):
            scores = {}
            result["scores"] = scores
        if not result.get("relatorio"):
            result["relatorio"] = gerar_relatorio(scores, tipo_entrada=tipo)
        return result

    # Fallback para pipeline antigo
    t0 = time.time()
    scores = predict_brain_activation(path)
    elapsed = time.time() - t0
    ux_score = int(round(float(np.mean(list(scores.values()))))) if scores else 0
    return {
        "scores": scores,
        "relatorio": gerar_relatorio(scores, tipo_entrada=tipo),
        "elapsed": elapsed,
        "ux_score": ux_score,
        "confidence_by_region": {},
        "evidence_by_region": {},
        "features": {},
    }


# =============================================
# FUNCAO PRINCIPAL — interface Gradio visual
# =============================================
def analisar(arquivo):
    path = extrair_path(arquivo)
    if path is None:
        return None, "Faca upload de uma imagem ou video."

    print(f"[analisar] path={path}")
    result = _run_analysis(path)
    scores = result.get("scores", {})
    elapsed = float(result.get("elapsed", 0.0))

    grafico = gerar_grafico(scores)
    relatorio_base = result.get("relatorio") or gerar_relatorio(scores, tipo_entrada=("video" if _is_video(path) else "imagem estatica"))
    relatorio = f"Inferencia: {elapsed:.1f}s\n\n" + relatorio_base
    return grafico, relatorio


# =============================================
# FUNCAO JSON — endpoint para o dashboard web
# =============================================
def analisar_json(arquivo):
    path = extrair_path(arquivo)
    if path is None:
        return json.dumps({"error": "no file"}, ensure_ascii=False)

    print(f"[analisar_json] path={path}")
    result = _run_analysis(path)

    payload = {
        "scores": result.get("scores", {}),
        "relatorio": result.get("relatorio", ""),
        "elapsed": float(result.get("elapsed", 0.0)),
        "ux_score": result.get("ux_score"),
        "confidence_by_region": result.get("confidence_by_region", {}),
        "evidence_by_region": result.get("evidence_by_region", {}),
        "features": result.get("features", {}),
    }
    return json.dumps(_to_builtin(payload), ensure_ascii=False)


# =============================================
# PUBLICAR URL NO GIST
# =============================================
def publicar_url_gist(url):
    if not GITHUB_TOKEN or not GIST_ID:
        print("Sem GITHUB_TOKEN ou GIST_ID — publicacao no Gist ignorada.")
        return
    try:
        payload = {"files": {"gradio_url.json": {"content": json.dumps({"url": url})}}}
        resp = requests.patch(
            f"https://api.github.com/gists/{GIST_ID}",
            headers={"Authorization": f"token {GITHUB_TOKEN}", "Accept": "application/vnd.github.v3+json"},
            json=payload,
        )
        if resp.status_code == 200:
            print(f"URL publicada no Gist: {url}")
        else:
            print(f"Erro ao atualizar Gist: {resp.status_code} {resp.text[:200]}")
    except Exception as e:
        print(f"Erro ao publicar no Gist: {e}")


# =============================================
# INTERFACE GRADIO
# =============================================
with gr.Blocks(title="NeuralUX — UX Brain Analyzer") as app:
    gr.Markdown("## NeuralUX — UX Brain Analyzer")
    gr.Markdown("Upload de **screenshot** ou **video de tela** para analisar a ativacao cerebral prevista.")

    with gr.Row():
        with gr.Column(scale=1):
            entrada = gr.File(label="Tela do app", file_types=[".png", ".jpg", ".jpeg", ".webp", ".mp4", ".mov", ".mkv", ".webm", ".avi", ".m4v"])
            botao = gr.Button("Analisar", variant="primary")
            gr.Markdown("*Tempo estimado: 5-25 segundos*")
        with gr.Column(scale=2):
            grafico_out = gr.Image(label="Ativacao por regiao cerebral")
            relatorio_out = gr.Textbox(label="Relatorio de UX", lines=20)

    # fn_index=0: interface visual
    botao.click(fn=analisar, inputs=[entrada], outputs=[grafico_out, relatorio_out])

    # fn_index=1: endpoint JSON para o dashboard
    json_input = gr.File(visible=False)
    json_output = gr.Textbox(visible=False)
    json_btn = gr.Button(visible=False)
    json_btn.click(fn=analisar_json, inputs=[json_input], outputs=[json_output])

# Launch e capturar a share URL corretamente
result = app.launch(share=True)

# Extrair share_url dependendo do formato de retorno
share_url = None
if isinstance(result, tuple):
    for item in result:
        if isinstance(item, str) and "gradio.live" in item:
            share_url = item
            break
elif hasattr(result, 'share_url'):
    share_url = result.share_url
if not share_url and hasattr(app, 'share_url'):
    share_url = app.share_url

if share_url:
    print(f"\nShare URL: {share_url}")
    publicar_url_gist(share_url)
else:
    print("\nNao foi possivel detectar a share URL automaticamente.")
    print("Cole a URL manualmente abaixo e rode:")
    print('  publicar_url_gist("https://XXXXX.gradio.live")')

